In [ ]:
import pandas as pd

In [ ]:
data_path = r"C:\Users\lucij\Desktop\Leiden\Year 2\Thesis Project\2024_data\combined_dataset.csv"
df = pd.read_csv(data_path)

# Demographics

In [ ]:
df.shape
print("There are this many participants:", df['subject'].nunique())

In [ ]:
demo_cols = [
    'subject',
    'age_slider.response',
    'mouse_5.clicked_name',   # gender
    'mouse_6.clicked_name'    # handedness
]

demo = (
    df[demo_cols]
    .groupby('subject', as_index=False)
    .first()
)

In [ ]:
demo['gender'] = (
    demo['mouse_5.clicked_name']
    .str.strip("[]'")
)

demo['handedness'] = (
    demo['mouse_6.clicked_name']
    .str.strip("[]'")
)

demo['age'] = pd.to_numeric(demo['age_slider.response'], errors='coerce')
demo = demo[['subject', 'age', 'gender', 'handedness']]

In [ ]:
age_mean = demo['age'].mean()
age_sd = demo['age'].std()
age_min = demo['age'].min()
age_max = demo['age'].max()

In [ ]:
gender_counts = demo['gender'].value_counts()
handedness_counts = demo['handedness'].value_counts()

In [ ]:
print("Age statistics:")
print(f"Mean: {age_mean}, SD: {age_sd}, Min: {age_min}, Max: {age_max}")

print("Gender counts:")
print(gender_counts)

print("Handedness counts:")
print(handedness_counts)

## By instructions

In [ ]:
instr = (
    df[['subject', 'instructions']]
    .groupby('subject', as_index=False)
    .first()
)

In [ ]:
demo = demo.merge(instr, on='subject')

In [ ]:
demo_instr0 = demo[demo['instructions'] == 0]
demo_instr1 = demo[demo['instructions'] == 1]

### Age

In [ ]:
demo.groupby('instructions')['age'].agg(['mean', 'std', 'min', 'max', 'count'])

### Gender

In [ ]:
pd.crosstab(demo['instructions'], demo['gender'])

### Handedness

In [ ]:
pd.crosstab(demo['instructions'], demo['handedness'])

### Differences

In [ ]:
from scipy.stats import ttest_ind

ttest_ind(
    demo_instr0['age'],
    demo_instr1['age'],
    equal_var=False
)

In [ ]:
from scipy.stats import chi2_contingency

chi2_contingency(
    pd.crosstab(demo['instructions'], demo['gender'])
)


### Session length

In [ ]:
import pandas as pd


datetime_format = "%Y-%m-%d_%Hh%M.%S.%f"

df["session_start_dt"] = pd.to_datetime(
    df["session_start"], format=datetime_format
)

df["session_end_dt"] = pd.to_datetime(
    df["session_end"], format=datetime_format
)


df["session_length_minutes"] = (
    (df["session_end_dt"] - df["session_start_dt"])
    .dt.total_seconds() / 60
)


session_df = (
    df[["subject", "instructions", "session_length_minutes"]]
    .drop_duplicates()
    .reset_index(drop=True)
)


overall_stats = {
    "mean_minutes": session_df["session_length_minutes"].mean(),
    "min_minutes": session_df["session_length_minutes"].min(),
    "max_minutes": session_df["session_length_minutes"].max(),
}

print("OVERALL SESSION LENGTH (minutes):")
for k, v in overall_stats.items():
    print(f"{k}: {v:.2f}")


condition_stats = (
    session_df
    .groupby("instructions")["session_length_minutes"]
    .agg(
        mean_minutes="mean",
        min_minutes="min",
        max_minutes="max"
    )
    .reset_index()
)

print("\nSESSION LENGTH BY CONDITION (minutes):")
print(condition_stats.round(2))


In [ ]:
from scipy.stats import ttest_ind


instr0_sessions = session_df.loc[session_df["instructions"] == 0, "session_length_minutes"]
instr1_sessions = session_df.loc[session_df["instructions"] == 1, "session_length_minutes"]


t_stat, p_val = ttest_ind(instr0_sessions, instr1_sessions, equal_var=False)

print(f"t = {t_stat:.3f}, p = {p_val:.4f}")


print(f"Instr 0 mean session length: {instr0_sessions.mean():.2f} min")
print(f"Instr 1 mean session length: {instr1_sessions.mean():.2f} min")

In [ ]:
check_df = (
    df[[
        "subject",
        "instructions",
        "session_start",
        "session_start_dt",
        "session_end",
        "session_end_dt",
        "session_length_minutes"
    ]]
    .drop_duplicates()
    .sort_values("subject")
    .reset_index(drop=True)
)

# Print everything
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print(check_df)